In [12]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np
import io

In [13]:
#========
# Setup
#========
df = pd.DataFrame({
    "order_id": ["101" , "102" , None , "104"] ,
    "revenue": ["1,200.50" , "950" , "N/A" , "1,050"] ,
    "region": ["East" , "West" , "East" , None] ,
    "vip_flag": ["True" , "False" , None , "True"] ,
    "notes": ["ok" , None , "follow up" , "ok"] ,
})
df

,order_id,revenue,region,vip_flag,notes
0,101,"1,200.50",East,True,ok
1,102,950,West,False,None
2,None,N/A,East,None,follow up
3,104,"1,050",None,True,ok


In [14]:
#=================
# Case 1) dtype
#=================
df.dtypes
print("\n")
df.info()
print("\n")
obj_cols = df.select_dtypes(include = ["object"]).columns.tolist()
obj_cols

order_id    object
revenue     object
region      object
vip_flag    object
notes       object
dtype: object



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   order_id  3 non-null      object
 1   revenue   4 non-null      object
 2   region    3 non-null      object
 3   vip_flag  3 non-null      object
 4   notes     3 non-null      object
dtypes: object(5)
memory usage: 292.0+ bytes




['order_id', 'revenue', 'region', 'vip_flag', 'notes']

In [15]:
#==============================================
# Case 2) astype() for explicit conversions
#==============================================
case2 = df.copy()
case2["notes"] = case2["notes"].astype("string")
case2["notes"].dtype

string[python]

In [16]:
#========================
# Case 3) to_numeric()
#========================
case3 = df.copy()
rev_clean = case3["revenue"].str.replace("," , "" , regex = False)
case3["revenue_num"] = pd.to_numeric(rev_clean , errors = "coerce")
case3["revenue_num_small"] = pd.to_numeric(rev_clean , errors = "coerce" , downcast = "float")
case3[["revenue" , "revenue_num" , "revenue_num_small"]]

,revenue,revenue_num,revenue_num_small
0,"1,200.50",1200.5,1200.5
1,950,950.0,950.0
2,N/A,NaN,NaN
3,"1,050",1050.0,1050.0


In [17]:
#=================================
# Case 4) Nullable integer dtype
#=================================
case4 = df.copy()
case4["order_id_int"] = pd.to_numeric(case4["order_id"] , errors = "coerce").astype("Int64")
case4[["order_id", "order_id_int"]]
case4["order_id_int"].dtype

,order_id,order_id_int
0,101,101
1,102,102
2,None,<NA>
3,104,104


Int64Dtype()

In [18]:
#=====================================================
# Case 5) String dtype ("string") + nullable outputs
#=====================================================
case5 = df.copy()
case5["notes"] = case5["notes"].astype("string")
case5["has_ok"] = case5["notes"].str.contains("ok")
case5["len_notes"] = case5["notes"].str.len()
case5[["notes", "has_ok", "len_notes"]]
case5[["has_ok", "len_notes"]].dtypes

,notes,has_ok,len_notes
0,ok,True,2
1,<NA>,<NA>,<NA>
2,follow up,False,9
3,ok,True,2


has_ok       boolean
len_notes      Int64
dtype: object

In [22]:
# ============================================================
# Case 6) PyArrow-backed dtypes
# Patterns:
# - Read with dtype_backend="pyarrow"
# - Convert with convert_dtypes(dtype_backend="pyarrow")
# - Cast with dtype="int64[pyarrow]"
# ============================================================
csv_data = io.StringIO("""order_id,revenue,region
101,1200.50,East
102,950,West
,1050,East
104,,West
""")

try:
    import pyarrow

    df_pa = pd.read_csv(csv_data, dtype_backend="pyarrow")
    df_pa.dtypes

    df_conv = df.copy().convert_dtypes(dtype_backend="pyarrow")
    print("\n")
    df_conv.dtypes

    ser_pa = pd.Series([1, 2, None], dtype="int64[pyarrow]")
    print("\n")
    ser_pa.dtype

except Exception as e:
    print("\nCase 6) PyArrow not available (skipping).")
    print(type(e).__name__, ":", e)

case_bonus = df.copy()
case_bonus["region_cat"] = case_bonus["region"].astype("category")
print("\n")
case_bonus[["region", "region_cat"]].dtypes

order_id     int64[pyarrow]
revenue     double[pyarrow]
region      string[pyarrow]
dtype: object

order_id    string[pyarrow]
revenue     string[pyarrow]
region      string[pyarrow]
vip_flag    string[pyarrow]
notes       string[pyarrow]
dtype: object

int64[pyarrow]

region          object
region_cat    category
dtype: object